# 11 Feature-Layer Ablation

**Question:** Which UNet layer provides the best feature representation for self-distillation?

We compare four extraction points:
- `bottleneck` — global avg-pool of mid-block output (original)
- `skip1` — last encoder skip connection
- `skip2` — second-to-last encoder skip
- `decoder1` — first decoder ResBlock output

Metric: FID (generation quality) + linear probe accuracy (representation quality).

In [ ]:
from src.experiments import load_cfg, deep_update, run_feature_layer_ablation
import matplotlib.pyplot as plt
import pandas as pd

base_cfg = load_cfg('configs/cifar10.yaml')
base_cfg = deep_update(base_cfg, {
    'wandb': {'enabled': True, 'tags': ['cifar10', 'feature_layer_ablation']},
})

results = run_feature_layer_ablation(base_cfg)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

results_sorted_fid = results.sort_values('fid')
axes[0].barh(results_sorted_fid['feature_layer'], results_sorted_fid['fid'], color='steelblue')
axes[0].set_xlabel('FID ↓')
axes[0].set_title('FID by feature layer')

results_sorted_acc = results.sort_values('linear_probe_acc', ascending=False)
axes[1].barh(results_sorted_acc['feature_layer'], results_sorted_acc['linear_probe_acc'], color='darkorange')
axes[1].set_xlabel('Linear probe acc ↑')
axes[1].set_title('Representation quality by feature layer')

plt.tight_layout()
plt.savefig('outputs/figures/feature_layer_ablation.png', dpi=150)
plt.show()
print('Best FID layer:     ', results.loc[results['fid'].idxmin(), 'feature_layer'])
print('Best probe-acc layer:', results.loc[results['linear_probe_acc'].idxmax(), 'feature_layer'])

**Key insight to look for:** If the best FID layer ≠ best probe-acc layer, the optimal
distillation target for generation quality differs from the one for representation quality.
This is a novel finding worth highlighting in the portfolio.